# Implementation of Deep Convolutional GANs
Reference: https://arxiv.org/pdf/1511.06434.pdf

---
# Learning guide: from neural networks to a DCGAN

This section was added for early learners. Read it first, then follow the short **"What this cell does"** note placed before each group of code cells.

## 1. What is a neural network?

A neural network is a function with many adjustable numbers (**parameters**, also called weights) that is trained from examples instead of being programmed with rules.

It is built from **layers**. A simple layer computes

$$a = f(Wx + b)$$

where $x$ is the input, $W$ and $b$ are learnable weights and biases, and $f$ is a non-linear **activation function** (ReLU, LeakyReLU, Tanh, Sigmoid, ...). Without the non-linearity, stacking layers would collapse into a single linear function and could not learn complex patterns.

Key vocabulary used in this notebook:

| Term | Meaning |
|---|---|
| **Tensor** | A multi-dimensional array (an image batch has shape `batch x channels x height x width`). |
| **Loss** | One number measuring how wrong the network currently is. |
| **Gradient** | How much the loss changes if a parameter changes slightly. |
| **Optimizer** | The rule that nudges parameters using gradients (here, Adam). |
| **Batch** | A small group of examples processed together (here, 64 images). |
| **Epoch** | One full pass over the training data. |
| **Device** | The hardware that does the math: CPU, NVIDIA GPU (CUDA), or Apple GPU (MPS). |

## 2. How does a network learn?

Training repeats four steps thousands of times:

1. **Forward pass:** feed a batch through the network to get outputs.
2. **Loss:** compare outputs with the targets.
3. **Backward pass (backpropagation):** `loss.backward()` computes the gradient of the loss with respect to every parameter.
4. **Update:** `optimizer.step()` moves each parameter a small step that reduces the loss. `zero_grad()` clears old gradients first, because PyTorch accumulates them.

## 3. Why convolutions for images?

Images have local structure: edges, corners, and textures are made from neighboring pixels. A **convolution** slides a small filter (here $3 \times 3$ or $4 \times 4$) over the image and reuses the same weights at every position, so it needs far fewer parameters than a fully connected layer and detects a pattern wherever it appears.

- `Conv2d` with `stride=2` **shrinks** the image (for example $32 \to 16$). The discriminator uses this to summarize an image into a single score.
- `ConvTranspose2d` with `stride=2` **enlarges** the image (for example $2 \to 4$). The generator uses this to grow a small feature map into a full image.

## 4. What is a GAN?

A **Generative Adversarial Network (GAN)** learns to *create* new data that looks like its training data. It consists of two networks that compete:

| Network | Role | Analogy |
|---|---|---|
| **Generator** $G$ | Turns random noise $z$ into a fake image | A forger painting fake banknotes |
| **Discriminator** $D$ | Outputs the probability that an image is real | A detective inspecting banknotes |

The **latent vector** $z$ (here 100 random numbers drawn from a standard normal distribution) is the generator's only input. Different random $z$ values produce different images. The generator is learning a mapping from simple noise to the space of realistic car pictures.

The classic GAN objective is a two-player game:

$$\min_G \max_D \; \mathbb{E}_{x \sim \text{real}}[\log D(x)] + \mathbb{E}_{z}[\log(1 - D(G(z)))]$$

In practice each training step does two things:

1. **Update $D$:** real images should score near 1, fake images near 0. The fake images are `detach()`-ed so this step changes only $D$.
2. **Update $G$:** produce fakes that $D$ scores near 1. The notebook does this by labeling fakes as "real" in the generator loss (the common *non-saturating* form of the generator loss, which gives stronger gradients early in training).

Neither network is told what a car looks like. $G$ improves only because $D$ keeps getting better at spotting its mistakes, and $D$ improves because $G$ keeps producing harder fakes.

## 5. What makes it a *Deep Convolutional* GAN (DCGAN)?

The DCGAN paper (Radford et al., 2015) gave a recipe that made GAN training much more reliable for images:

- Use **strided convolutions** to downsample in $D$ and **transposed convolutions** to upsample in $G$ (no max-pooling).
- Use **batch normalization** in both networks (but not on $D$'s first layer or $G$'s output layer).
- Use **ReLU** in $G$ with a **Tanh** output, and **LeakyReLU** in $D$.
- Train with **Adam**, learning rate 0.0002, $\beta_1 = 0.5$, and initialize weights from $\mathcal{N}(0, 0.02)$.

This notebook now applies these points to the discriminator (4x4 strided convolutions, widths 64 -> 512, no BatchNorm on the first layer, a convolutional head), to both optimizers (learning rate 0.0002) and to both networks (`weights_init`). The generator is not yet fully paper-style: it starts with a `Linear` layer and an extra $3 \times 3$ convolution instead of going straight into transposed convolutions.

## 6. How to read the training numbers

- Both networks use binary cross-entropy. A coin-flip guess ($D = 0.5$) costs $\ln 2 \approx 0.693$ per term.
- `D_loss` is the sum of a real term and a fake term, so a perfectly balanced discriminator sits near $2 \ln 2 \approx 1.386$. An earlier run of this notebook (with the previous discriminator, learning rate, and default initialization) logged `D Loss: 1.4036` and `G Loss: 0.7998` at step 0, which is what two untrained networks look like. Expect different numbers now.
- GAN losses are **not** like classifier accuracy. They rarely decrease smoothly. The most reliable check is to **look at the generated samples** as training progresses.
- Warning signs: `D_loss` collapsing toward 0 while `G_loss` keeps growing (the detective is winning too easily), or the generator producing nearly identical images for different noise (**mode collapse**).


## Part 1 - Setting up the toolbox (the environment check and import cells)

**Run this notebook in an Anaconda environment.** On the Mac mini the repository's environment file builds one:

```bash
conda env create -f CONDA_ENV_TORCH_VISION_AUDIO-PYTHON3.11-MAC-ARM64.YML
conda activate testdrive-dl-mm
```

Then select the `testdrive-dl-mm` interpreter as the notebook kernel (VS Code: **Select Kernel** -> **Python Environments**). Run the commands in macOS Terminal, never in a notebook cell, and restart the kernel after installing or changing packages. Keep one source for PyTorch in the environment; do not mix a conda build and a pip build of `torch`.

The next seven code cells only check the environment and load libraries. Nothing is trained yet. Each cell has a small job:

| Code cell | What it does | Why it is needed |
|---|---|---|
| Environment check | Prints the Python version, CPU architecture, interpreter path, conda environment name, and the versions of the key packages | Confirms the kernel is the Anaconda environment you intended (for example `arm64` on Apple Silicon) before anything else runs. It lists any missing package. |
| Core PyTorch | `torch`, `torchvision`, `torch.nn`, `torch.nn.functional` | `torch` provides tensors and automatic differentiation. `torch.nn` provides layers (`Conv2d`, `Linear`, ...) and losses. |
| Data tools | `DataLoader`, `Dataset`, `datasets`, `transforms`, `save_image` | `Dataset` + `DataLoader` serve images in shuffled mini-batches. `transforms` resizes and normalizes images. |
| Utilities | `numpy`, `datetime`, `os`, `sys` | Array math, timestamps for logs, folder creation for saved samples. |
| `glob` | `glob` | Finds files by pattern (for example all `*.jpg` files in a folder). |
| `PIL` | `Image` | Opens image files from disk. |
| Matplotlib | `imshow`, `imsave` | Shows images in the notebook and writes sample images to disk. `%matplotlib inline` makes plots appear below the cell. |

Some imports (`F`, `Dataset`, `datasets`, `save_image`, `np`, `sys`, `glob`) are not used directly in this notebook. They are leftovers from the original tutorial and are harmless.


In [ ]:
# Environment check: confirm the kernel runs in the intended Anaconda environment.
# Packages are installed from a terminal with the environment activated, not from this notebook.
import os, sys, platform
from importlib import metadata

print("Python:", sys.version.split()[0], "|", platform.system(), platform.machine())
print("Executable:", sys.executable)
print("Conda env:", os.environ.get("CONDA_DEFAULT_ENV") or os.path.basename(sys.prefix))
if platform.system() == "Darwin" and platform.machine() != "arm64":
    print("WARNING: not a native Apple Silicon (arm64) Python; MPS will not be available.")

missing = []
for pkg in ("torch", "torchvision", "numpy", "pillow", "matplotlib", "tqdm"):
    try:
        print(f"{pkg}: {metadata.version(pkg)}")
    except metadata.PackageNotFoundError:
        missing.append(pkg)
if missing:
    print("MISSING in this kernel's environment:", ", ".join(missing))
    print("Install them in a terminal with this environment activated, then restart the kernel.")


In [ ]:
import torch
import torchvision
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
from torch.utils.data import DataLoader
from torch.utils.data.dataset import Dataset
from torchvision import datasets
from torchvision import transforms
from torchvision.utils import save_image

In [ ]:
import numpy as np
import datetime
import os, sys

In [ ]:
import glob

In [ ]:
from PIL import Image

In [ ]:
from matplotlib.pyplot import imshow, imsave
%matplotlib inline

## Part 2 - Choosing the hardware

**What this cell does:** names the model and picks the compute device. It prefers Apple's GPU (**MPS**), then NVIDIA (**CUDA**), then the **CPU**, and prints what it found.

**Learn from it:**
- A model only runs on a device if its parameters *and* its input tensors have been moved there with `.to(DEVICE)`. A mismatch raises an error.
- MPS accelerates PyTorch on Apple Silicon. It is not the Neural Engine.
- `DEVICE` is defined once and reused everywhere below, so changing hardware means editing one cell.
- If it prints `Selected device: cpu` on the Mac mini, check the environment-check output from Part 1: the kernel must be a native `arm64` Python from the Anaconda environment, with a PyTorch build that includes MPS (`MPS built: True`).


In [ ]:
MODEL_NAME = 'DCGAN'

# Prefer Apple MPS on Mac hardware, then fall back to CUDA, then CPU.
# This follows the repo's MPS setup guidance and keeps the notebook portable.
has_mps = hasattr(torch.backends, 'mps')
print("PyTorch:", torch.__version__)
print("MPS built:", torch.backends.mps.is_built() if has_mps else False)
print("MPS available:", torch.backends.mps.is_available() if has_mps else False)

if has_mps and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
else:
    DEVICE = torch.device("cpu")

print(f"Selected device: {DEVICE}")

## Part 3 - Defining the problem size

**What this cell does:** `IMAGE_DIM = (32, 32, 3)` says the model works with $32 \times 32$ pixel colour images (3 = red, green, blue).

**Learn from it:** this one setting decides the shapes of everything else. The generator must output $32 \times 32$ images, the discriminator must accept them, and the preprocessing step resizes the real photos to match. Small images train quickly but cannot show fine detail, so expect blurry, car-like blobs rather than sharp photographs.

In [ ]:
IMAGE_DIM = (32, 32, 3)

## Part 4 - A helper to look at the generator's output

**What this cell does:** `get_sample_image(G, n_noise)` draws 10 random noise vectors, runs them through the generator, and converts the result into something Matplotlib can display.

**Step by step:**
1. `torch.randn(10, n_noise, device=DEVICE)` creates 10 latent vectors directly on the compute device.
2. `G(z)` produces images shaped `(10, 3, 32, 32)` (channels first, PyTorch style).
3. `.permute(0, 2, 3, 1)` reorders to `(10, 32, 32, 3)` (channels last, Matplotlib style).
4. `(x + 1) / 2` maps pixel values from the generator's Tanh range $[-1, 1]$ back to the displayable range $[0, 1]$.

**Learn from it:** `.detach().cpu().numpy()` is the standard way to move a tensor out of the training graph and off the GPU before using it with NumPy or Matplotlib. (The docstring says "100 images" but the function actually returns 10.)

In [ ]:
def get_sample_image(G, n_noise):
    """
        save sample 100 images
    """
    # Construct latent vectors directly on the selected runtime device.
    z = torch.randn(10, n_noise, device=DEVICE)
    # Previously, this assumed output shape (10, 3, 28, 28), which does not match the 32x32 generator output.
    # Permuting without reshaping preserves the actual height and width, avoiding the element-count mismatch.
    y_hat = G(z).permute(0, 2, 3, 1)
    result = (y_hat.detach().cpu().numpy()+1)/2.
    return result

## Part 5 - The Discriminator (the detective)

**What this cell does:** defines a convolutional classifier that takes an image and outputs one number between 0 (fake) and 1 (real). It follows the DCGAN paper's discriminator design.

**Shape walk-through for a $3 \times 32 \times 32$ input:**

| Layer | Output shape | Notes |
|---|---|---|
| `Conv2d(3, 64, 4, stride=2)` + `LeakyReLU` | $64 \times 16 \times 16$ | Learns low-level patterns such as edges. **No BatchNorm** on this first layer |
| `Conv2d(64, 128, 4, stride=2)` + `BatchNorm` + `LeakyReLU` | $128 \times 8 \times 8$ | Combines edges into textures |
| `Conv2d(128, 256, 4, stride=2)` + `BatchNorm` + `LeakyReLU` | $256 \times 4 \times 4$ | Shapes and parts |
| `Conv2d(256, 512, 4, stride=2)` + `BatchNorm` + `LeakyReLU` | $512 \times 2 \times 2$ | Higher-level structure |
| `Conv2d(512, 1, kernel=2)` + `Sigmoid` | $1 \times 1 \times 1$ | Convolutional head: one kernel covers the whole $2 \times 2$ map and gives the probability that the image is real |

**Learn from it:**
- The widths **grow** with depth (64 -> 512) while the spatial size shrinks. This is the usual convolutional pattern: fewer positions, more feature channels.
- Kernel size 4 with stride 2 and padding 1 halves the size exactly and avoids the uneven overlap that a $3 \times 3$ kernel with stride 2 produces.
- `BatchNorm2d` stabilizes training, but the paper leaves it off the discriminator's first layer so the network can see the raw input statistics. `LeakyReLU(0.2)` lets a small gradient through for negative values, which helps GAN discriminators.
- `bias=False` is used on every convolution. Where `BatchNorm` follows, it has its own shift parameter. The final head is a single score squashed by the `Sigmoid`, so a bias adds little.
- The head replaces the earlier average-pooling + `Linear` layer. A convolution over the final map keeps the spatial layout instead of averaging it away. Its kernel size is computed as `IMAGE_DIM[0] // 16`, so it adapts to the image size.
- `Sigmoid` squashes the score into the range $(0, 1)$, which is what `BCELoss` expects.
- The `num_classes` and `y` arguments are unused leftovers kept for compatibility.


In [ ]:
class Discriminator(nn.Module):
    """
        Convolutional Discriminator following the DCGAN recipe (Radford et al., 2015):
        strided 4x4 convolutions with widths 64 -> 128 -> 256 -> 512, no BatchNorm on the
        first layer, LeakyReLU(0.2), and a convolutional head that outputs one probability.
    """
    def __init__(self, in_channel=3, num_classes=1):
        super(Discriminator, self).__init__()
        # Four stride-2 convolutions shrink the image by 2**4 = 16 (32 -> 2 for 32x32 inputs).
        final_size = IMAGE_DIM[0] // 2**4
        self.conv = nn.Sequential(
            # 32 -> 16 (no BatchNorm on the first layer)
            nn.Conv2d(in_channel, 64, 4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2),
            # 16 -> 8
            nn.Conv2d(64, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2),
            # 8 -> 4
            nn.Conv2d(128, 256, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2),
            # 4 -> 2
            nn.Conv2d(256, 512, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2),
        )
        # Convolutional head: one kernel covering the whole final feature map -> 1x1 score.
        self.head = nn.Sequential(
            nn.Conv2d(512, 1, final_size, stride=1, padding=0, bias=False),
            nn.Sigmoid(),
        )

    def forward(self, x, y=None):
        y_ = self.conv(x)
        y_ = self.head(y_)
        return y_.view(y_.size(0), -1)


## Part 6 - The Generator (the forger)

**What this cell does:** defines a network that turns a 100-number noise vector into a $3 \times 32 \times 32$ image.

**Shape walk-through:**

| Layer | Output shape | Notes |
|---|---|---|
| `Linear(100, 2*2*512)` + `ReLU` | $2048$ | `init_dim = 32 // 2**4 = 2`, so the starting map is $2 \times 2$ |
| `view(...)` | $512 \times 2 \times 2$ | Reshape the vector into a tiny feature map |
| `Conv2d(512, 512, 3, padding=1)` | $512 \times 2 \times 2$ | Refines features without changing size |
| `ConvTranspose2d(512, 256, 4, stride=2)` | $256 \times 4 \times 4$ | Each transposed conv doubles height and width |
| `ConvTranspose2d(256, 128, ...)` | $128 \times 8 \times 8$ | |
| `ConvTranspose2d(128, 128, ...)` | $128 \times 16 \times 16$ | |
| `ConvTranspose2d(128, 3, ...)` + `Tanh` | $3 \times 32 \times 32$ | Final RGB image with values in $[-1, 1]$ |

**Learn from it:**
- The `assert` checks that the image size is divisible by $2^4 = 16$, because the generator doubles the size four times.
- The generator is the mirror image of the discriminator: one shrinks images, the other grows them.
- `Tanh` outputs $[-1, 1]$, matching the normalization applied to the real images later, so real and fake images live in the same value range.

In [ ]:
class Generator(nn.Module):
    """
        Convolutional Generator for MNIST
    """
    def __init__(self, out_channel=1, input_size=100, num_classes=784):
        super(Generator, self).__init__()
        assert IMAGE_DIM[0] % 2**4 == 0, 'Should be divided 16'
        self.init_dim = (IMAGE_DIM[0] // 2**4, IMAGE_DIM[1] // 2**4)
        self.fc = nn.Sequential(
            nn.Linear(input_size, self.init_dim[0]*self.init_dim[1]*512),
            nn.ReLU(),
        )
        self.conv = nn.Sequential(
            nn.Conv2d(512, 512, 3, padding=1, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(),
            # x2
            nn.ConvTranspose2d(512, 256, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            # x2
            nn.ConvTranspose2d(256, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # x2
            nn.ConvTranspose2d(128, 128, 4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            # x2
            nn.ConvTranspose2d(128, out_channel, 4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )
        
    def forward(self, x, y=None):
        x = x.view(x.size(0), -1)
        y_ = self.fc(x)
        y_ = y_.view(y_.size(0), 512, self.init_dim[0], self.init_dim[1])
        y_ = self.conv(y_)
        return y_

## Part 7 - The data: Stanford Cars

**What the next cell does:** imports the `CARS` dataset class from `cars_dataset.py`, a small helper file next to this notebook.

**What that class does:**
- Finds every `*.jpg` in the data folder with `glob` and keeps the sorted file list.
- Removes a hard-coded list of 18 images that are grayscale. Grayscale images have one channel, which would break the 3-channel normalization used below.
- `__getitem__` opens one image with PIL and applies the transform; `__len__` reports how many images there are. This is the minimal contract PyTorch needs from a `Dataset`.

You must download the dataset yourself from the Stanford Cars page mentioned in the helper file's docstring.

In [ ]:
from cars_dataset import CARS

## Part 8 - Creating the two networks

**What this cell does:** defines `weights_init`, builds one `Discriminator` and one `Generator`, moves both to `DEVICE`, and applies the DCGAN weight initialization to both. The `in_channel` / `out_channel` argument is `IMAGE_DIM[-1]`, which is 3 (RGB).

**Learn from it:**
- PyTorch's default initialization is not what the DCGAN paper used. The paper draws all convolution weights from a normal distribution $\mathcal{N}(0, 0.02)$. `BatchNorm` scales are drawn from $\mathcal{N}(1, 0.02)$ and its shifts start at 0. `net.apply(weights_init)` calls the function on every layer of the network, and the function picks the rule by layer type.
- Small, similar starting weights keep both networks balanced at the start, so neither wins the game immediately.
- At this point both networks have **random weights**, so the discriminator guesses and the generator outputs noise. The commented-out `load_state_dict` lines show how you would resume from saved weights instead; loading them replaces the initialization.


In [ ]:
def weights_init(m):
    """
        DCGAN weight initialization: conv/linear weights ~ N(0, 0.02),
        BatchNorm scale ~ N(1, 0.02) and BatchNorm shift = 0.
    """
    classname = m.__class__.__name__
    if 'Conv' in classname or 'Linear' in classname:
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif 'BatchNorm' in classname:
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

D = Discriminator(in_channel=IMAGE_DIM[-1]).to(DEVICE)
G = Generator(out_channel=IMAGE_DIM[-1]).to(DEVICE)
D.apply(weights_init)
G.apply(weights_init)
# D.load_state_dict('D_dc.pkl')
# G.load_state_dict('G_dc.pkl')


## Part 9 - Preparing the images

**What this cell does:** builds the preprocessing pipeline applied to every photo:

1. `Resize((32, 32))` shrinks the image to the model's input size.
2. `ToTensor()` converts a PIL image (values 0-255) to a tensor with values in $[0, 1]$, shaped `channels x height x width`.
3. `Normalize(mean=0.5, std=0.5)` computes $(x - 0.5) / 0.5$, mapping values to $[-1, 1]$.

**Learn from it:** step 3 is essential for GANs. The generator ends with `Tanh` (range $[-1, 1]$), so real images must use the same range, otherwise the discriminator could tell real from fake just by the value range.

In [ ]:
transform = transforms.Compose([transforms.Resize((IMAGE_DIM[0],IMAGE_DIM[1])),
                                transforms.ToTensor(),
                                transforms.Normalize(mean=(0.5, 0.5, 0.5),
                                std=(0.5, 0.5, 0.5))
                               ]
)

## Part 10 - Loading the dataset

**What this cell does:** creates the `CARS` dataset object. The folder is read from the `CARS_DATA_DIR` environment variable, with `cars_train` in the current folder as the fallback, and the path is printed so you can confirm it.

**Learn from it:** keeping the path out of the code makes the notebook work on any machine. If you get an empty dataset or a file error, this is the first place to check.

In [ ]:
# Use an environment override when available so the notebook works on other machines.
DATA_PATH = os.environ.get('CARS_DATA_DIR', os.path.join(os.getcwd(), 'cars_train'))
dataset = CARS(data_path=DATA_PATH, transform=transform)
print(f"Using dataset path: {DATA_PATH}")

## Part 11 - Batching the data

**What the next two cells do:**
- `batch_size = 64` sets how many images are processed per training step.
- `DataLoader(...)` wraps the dataset so it hands out shuffled batches of 64 images. `drop_last=True` discards a final smaller batch, which matters because the label tensors later have a fixed size of 64. `num_workers=8` loads images in 8 background processes.

**Learn from it:**
- Training on mini-batches gives a good gradient estimate while fitting in memory, and shuffling prevents the network from learning the file order.
- Each batch has shape `(64, 3, 32, 32)`.
- On macOS, worker processes are started with `spawn`, which adds start-up time. If data loading seems stuck in a notebook, try a smaller `num_workers` (even 0) to rule it out.

In [ ]:
batch_size = 64

In [ ]:
data_loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True, drop_last=True, num_workers=8)

## Part 12 - Loss function and optimizers

**What this cell does:** chooses how mistakes are measured and how the networks are updated.

- `nn.BCELoss()` is **binary cross-entropy**: $-[y \log p + (1-y)\log(1-p)]$. It is small when the predicted probability $p$ is close to the label $y$ (1 = real, 0 = fake).
- Two separate **Adam** optimizers, one for `D.parameters()` and one for `G.parameters()`. Each only updates its own network.
- `lr=2e-4` (0.0002) is the learning rate recommended by the DCGAN paper. The earlier value of 0.001 is five times larger, which tends to make GAN training oscillate.
- `betas=(0.5, 0.999)`: the lower first momentum value (0.5) is the DCGAN recommendation for more stable GAN training.

**Try it:** if training is unstable (losses jumping wildly, noisy images), lower the learning rate further (for example `1e-4`). If it is very slow to improve, a modest increase is worth a try, but change one setting at a time.


In [ ]:
criterion = nn.BCELoss()
D_opt = torch.optim.Adam(D.parameters(), lr=2e-4, betas=(0.5, 0.999))
G_opt = torch.optim.Adam(G.parameters(), lr=2e-4, betas=(0.5, 0.999))


## Part 13 - Training settings and labels

**What the next two cells do:**
- `max_epoch = 100`: number of passes over the dataset. `step` counts total batches processed. `n_critic = 1` controls how often the generator is updated (every `n_critic` steps). `n_noise = 100` is the size of the latent vector.
- `VERBOSE` and `LOG_EVERY` control the progress display of the training cell. With `VERBOSE = False` you see a progress bar and one summary line per epoch. Set `VERBOSE = True` to bring back step-by-step debug messages, which help when training seems stuck. `LOG_EVERY = 20` means the bar's statistics are refreshed every 20 steps. Each refresh has to wait for the GPU to finish its queued work, so a very small value slows training slightly.
- `D_labels` is a column of ones (the target for "real") and `D_fakes` is a column of zeros (the target for "fake"), one entry per image in the batch, created on `DEVICE`.

**Learn from it:** GAN training is just two binary classification problems sharing the same discriminator. Labels are fixed; what changes is *which images* get which label.


In [ ]:
max_epoch = 100
step = 0
n_critic = 1 # for training more k steps about Discriminator
n_noise = 100

# Progress display settings for the training cell.
VERBOSE = False   # True prints the old step-by-step debug messages (useful when something seems stuck)
LOG_EVERY = 20    # refresh the progress-bar statistics every N steps (each refresh waits for the GPU)


In [ ]:
D_labels = torch.ones([batch_size, 1]).to(DEVICE) # Discriminator Label to real
D_fakes = torch.zeros([batch_size, 1]).to(DEVICE) # Discriminator Label to fake

## Part 14 - The training loop (the heart of the GAN)

**What this cell does:** for every epoch and every batch of real images it performs two updates.

**A. Train the discriminator**
1. Score the real batch `D(x)` and compare with the label 1 -> `D_x_loss`.
2. Sample noise `z`, generate `G(z)`, and `.detach()` it so no gradient flows into the generator. Score it and compare with the label 0 -> `D_z_loss`.
3. `D_loss = D_x_loss + D_z_loss`, then `zero_grad()`, `backward()`, `D_opt.step()`.

**B. Train the generator**
1. Sample fresh noise and compute `D(G(z))`. Here the fakes are *not* detached, so gradients flow back through $D$ into $G$.
2. Compare with the label **1** ("real"). The generator is rewarded when $D$ is fooled.
3. `G_loss.backward()` and `G_opt.step()` update only the generator's parameters.

**C. Progress display**
- **Progress bar:** one bar per epoch shows how many of the 126 batches are done, the speed, and the time left in the epoch. Its right-hand side shows the average of the last `LOG_EVERY` steps:
  - `D_loss` and `G_loss`: the two losses.
  - `Dx`: $D(x)$, the average score the discriminator gives real images.
  - `DGz`: $D(G(z))$, the average score it gives fake images.
- **Epoch summary line:** when an epoch finishes, the bar disappears and one line stays, with the epoch's average losses and scores, the epoch time, the speed in images per second, the total elapsed time, and the estimated time remaining (ETA).
- **Sample images:** every 500 steps it switches the generator to `eval()` mode, saves one sample image into a `samples/` folder, and switches back to `train()`.

**How to read the numbers:**
- A healthy, balanced game has `Dx` somewhat above 0.5 and `DGz` somewhat below it, both drifting towards 0.5 as the generator improves. Typical values are around 0.6 to 0.8 for `Dx` and 0.2 to 0.4 for `DGz`.
- `Dx` near 1 with `DGz` near 0 means the detective is winning easily and `G_loss` will grow. `Dx` and `DGz` both near 0.5 from the very start can also hide a problem, so keep looking at the sample images.
- The ETA is the average time of the epochs finished so far times the number of epochs left. Epoch 1 is often slower because data-loading workers start up, so the first ETA is pessimistic.
- The statistics are collected on the GPU and read back only every `LOG_EVERY` steps. This keeps the display from slowing training, because reading a value forces the GPU to finish its queued work.

**Things to notice:**
- `eval()` / `train()` matters because `BatchNorm` behaves differently in the two modes.
- The generator never sees a real image. It only learns from the discriminator's feedback.
- This cell can run for a long time (100 epochs). You can interrupt it with the stop button and still inspect and save the networks as they are (but note the checkpoint cell near the end only saves the final state; there is no automatic checkpoint during training yet).
- If the discriminator's loss falls close to 0 while the generator's keeps rising, one network is overpowering the other; revisit the learning rate or the update ratio (`n_critic`).


In [ ]:
import time
from tqdm.auto import tqdm

def fmt_time(seconds):
    """Format seconds as H:MM:SS."""
    return str(datetime.timedelta(seconds=int(seconds)))

train_start = time.perf_counter()
epoch_times = []
zero = torch.zeros((), device=DEVICE)

for epoch in range(max_epoch):
    if VERBOSE:
        print(f"Starting epoch {epoch + 1}/{max_epoch}; waiting for batches", flush=True)

    # Running statistics are accumulated as tensors on DEVICE and only read back
    # (.tolist()) every LOG_EVERY steps and at the end of the epoch. Calling .item()
    # on every step would force the GPU to finish its queued work each time.
    # Order of the 4 values: D_loss, G_loss, D(x), D(G(z)).
    ep_sum = torch.zeros(4, device=DEVICE)
    win_sum = torch.zeros(4, device=DEVICE)
    ep_g_n = win_g_n = win_n = 0
    epoch_start = time.perf_counter()

    with tqdm(data_loader, desc=f"Epoch {epoch + 1}/{max_epoch}", leave=False) as bar:
        for idx, images in enumerate(bar):
            debug_step = VERBOSE and step % 100 == 0
            if VERBOSE and (idx == 0 or debug_step):
                print(f"Batch received: epoch={epoch + 1}, batch={idx + 1}/{len(data_loader)}, step={step}, shape={tuple(images.shape)}", flush=True)
            # Training Discriminator:
            # 1) Real images should be classified as real.
            # 2) Fake images generated by the generator should be classified as fake.
            # We detach the fake samples here so that the discriminator update does not
            # create a computational graph that backpropagates into the generator.
            x = images.to(DEVICE)
            if debug_step:
                print("Discriminator update started", flush=True)
            x_outputs = D(x)
            D_x_loss = criterion(x_outputs, D_labels)
            D_x_score = x_outputs.detach().mean()  # D(x): average score on real images

            z = torch.randn(batch_size, n_noise, device=DEVICE)
            fake_images = G(z).detach()  # stop gradients to generator during D update
            z_outputs = D(fake_images)
            D_z_loss = criterion(z_outputs, D_fakes)
            D_z_score = z_outputs.detach().mean()  # D(G(z)): average score on fake images
            D_loss = D_x_loss + D_z_loss

            D.zero_grad()
            D_loss.backward()
            D_opt.step()
            if debug_step:
                print(f"Discriminator update complete: loss={D_loss.item():.4f}", flush=True)

            # Training Generator:
            # We want the generator to produce samples that fool the discriminator,
            # so we label generated images as real for the generator loss.
            # The discriminator is used only for scoring the generator output, and its
            # parameters are not updated in this branch.
            g_updated = step % n_critic == 0
            if g_updated:
                z = torch.randn(batch_size, n_noise, device=DEVICE)
                if debug_step:
                    print("Generator update started (generator/discriminator forward pass)", flush=True)
                z_outputs = D(G(z))
                G_loss = criterion(z_outputs, D_labels)

                D.zero_grad()
                G.zero_grad()
                G_loss.backward()
                G_opt.step()
                if debug_step:
                    print(f"Generator update complete: loss={G_loss.item():.4f}", flush=True)

            # Accumulate statistics without synchronizing with the GPU.
            stats = torch.stack([D_loss.detach(), G_loss.detach() if g_updated else zero, D_x_score, D_z_score])
            ep_sum += stats
            win_sum += stats
            win_n += 1
            if g_updated:
                ep_g_n += 1
                win_g_n += 1

            # Refresh the progress bar with the average of the last LOG_EVERY steps.
            if win_n == LOG_EVERY:
                v = win_sum.tolist()
                bar.set_postfix(D_loss=f"{v[0] / win_n:.3f}",
                                G_loss=f"{v[1] / max(win_g_n, 1):.3f}",
                                Dx=f"{v[2] / win_n:.2f}",
                                DGz=f"{v[3] / win_n:.2f}")
                win_sum.zero_()
                win_n = win_g_n = 0

            if step % 500 == 0:
                G.eval()
                img = get_sample_image(G, n_noise)
                os.makedirs('samples', exist_ok=True)
                imsave('samples/{}_step{:05d}.jpg'.format(MODEL_NAME, step), img[0])
                G.train()
            step += 1

    # End of epoch: .tolist() waits for the device, so the timing below is accurate.
    ep = ep_sum.tolist()
    epoch_time = time.perf_counter() - epoch_start
    epoch_times.append(epoch_time)
    n_steps = len(data_loader)
    elapsed = time.perf_counter() - train_start
    eta = (sum(epoch_times) / len(epoch_times)) * (max_epoch - epoch - 1)
    print(f"Epoch {epoch + 1:>3}/{max_epoch} | "
          f"D {ep[0] / n_steps:.3f}  G {ep[1] / max(ep_g_n, 1):.3f} | "
          f"D(x) {ep[2] / n_steps:.2f}  D(G(z)) {ep[3] / n_steps:.2f} | "
          f"{epoch_time:.1f}s ({n_steps * batch_size / epoch_time:.0f} img/s) | "
          f"elapsed {fmt_time(elapsed)}  ETA {fmt_time(eta)}", flush=True)


## Sample

### Part 15 - Looking at results

**What the next two cells do:**
- The first switches the generator to `eval()` mode and displays one generated image from random noise.
- The second opens real image number 999 from the dataset, applies the same preprocessing, and shows it. `(x + 1) / 2` undoes the normalization so it displays correctly.

**Learn from it:** compare the two. Early in training the generated image looks like coloured static. Later it should show car-like shapes and colours. At $32 \times 32$ resolution, do not expect photographic detail. Re-run the first cell several times: each run uses new noise and should give a different image. If every run gives nearly the same picture, that is **mode collapse**.

In [ ]:
# generation to image
G.eval()
imshow(get_sample_image(G, n_noise)[0])

In [ ]:
# Real Image
t = Image.open(dataset.fpaths[999])
t = (transform(t).permute(1, 2, 0)+1)/2.
imshow(t)

### Part 16 - Saving your work

**What the next two cells do:**
- `save_checkpoint(state, file_name)` is a small wrapper around `torch.save`.
- The second cell saves a dictionary containing the epoch number, both networks' weights (`state_dict()`), and both optimizers' states to `dcgan.pth.tar`.

**Learn from it:** a **state dict** holds only the learned numbers, not the code, so the class definitions must exist when you load it back. Saving optimizer state as well lets you resume training exactly where you stopped. To reload later: build `D` and `G`, then call `D.load_state_dict(ckpt['D'])` and `G.load_state_dict(ckpt['G'])`.

In [ ]:
def save_checkpoint(state, file_name='checkpoint.pth.tar'):
    torch.save(state, file_name)

In [ ]:
# Saving params.
# torch.save(D.state_dict(), 'D_c.pkl')
# torch.save(G.state_dict(), 'G_c.pkl')
save_checkpoint({'epoch': epoch + 1,
                 'D':D.state_dict(),
                 'G':G.state_dict(),
                 'd_optim': D_opt.state_dict(),
                 'g_optim' : G_opt.state_dict()},
                'dcgan.pth.tar')